<a href="https://colab.research.google.com/github/trobinso4124-alt/brandnewnotebook12/blob/main/video_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import subprocess
import os
import time
import urllib.request

# --- 1. Clone the BGutil POT provider (if not already cloned) ---
if not os.path.isdir("/tmp/bgutil"):
    print("📥 Cloning BGutil POT provider...")
    subprocess.run(
        ["git", "clone", "https://github.com/Brainicism/bgutil-ytdlp-pot-provider.git", "/tmp/bgutil"],
        check=False,
    )

# --- 2. Install Node dependencies and build ---
server_dir = "/tmp/bgutil/server"
if os.path.isdir(server_dir):
    print("🔨 Building BGutil server...")
    subprocess.run(["npm", "install", "--production"], cwd=server_dir, check=False)
    subprocess.run(["npm", "install", "typescript"], cwd=server_dir, check=False)
    subprocess.run(["./node_modules/.bin/tsc"], cwd=server_dir, check=False)

    # --- 3. Launch the POT server in the background ---
    # IMPORTANT: use Popen, not ! with & — Jupyter doesn't allow & backgrounding.
    print("🚀 Starting BGutil POT server on port 4416...")
    subprocess.Popen(
        ["node", "build/main.js", "--port", "4416"],
        cwd=server_dir,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    # Give it a moment to boot
    time.sleep(5)

    # --- 4. Verify the server is alive ---
    try:
        with urllib.request.urlopen("http://127.0.0.1:4416/ping", timeout=5) as r:
            print(f"✅ BGutil POT server is up (status {r.status}).")
    except Exception as e:
        print(f"⚠️ Could not reach POT server: {e}")
        print("   Check the Node version (see below).")
else:
    print("❌ /tmp/bgutil/server not found — clone may have failed.")

In [ ]:
import os
import shutil
import subprocess
import ipywidgets as widgets
from IPython.display import display, clear_output, Video, FileLink
import yt_dlp

print("🚀 Video search ready!")

# ---------- Widgets ----------
search_input = widgets.Text(
    placeholder="Type video name or topic...",
    description="🎥 Search:",
    layout=widgets.Layout(width="65%")
)

search_button = widgets.Button(
    description="Find Video",
    button_style="info",
    icon="search"
)

run_button = widgets.Button(
    description="Download & Play",
    button_style="success",
    icon="play",
    disabled=True
)

track_select = widgets.Dropdown(
    options=[],
    description="🔢 Video:",
    disabled=True,
    layout=widgets.Layout(width="65%")
)

output_area = widgets.Output()

# ---------- Cookie setup ----------
COOKIE_FILE = "cookies.txt"

def get_cookie_opt():
    """Return a dict with cookiefile option if cookies.txt exists, else empty dict."""
    if os.path.exists(COOKIE_FILE):
        return {"cookiefile": COOKIE_FILE}
    return {}

# ---------- Shared yt-dlp options (bot-detection bypass) ----------
def get_base_opts():
    """
    Options shared by search and download that help bypass YouTube's
    datacenter-IP bot check.
    """
    opts = {
        # Use the android_vr client: JS-less, usually skips the LOGIN_REQUIRED check
        "extractor_args": {
            "youtube": {
                "player_client": ["android_vr"],
            }
        },
        # Node.js runtime for any remaining JS challenges (correct dict format)
        "js_runtimes": {"node": {}},
        # Datacenter-friendly network settings
        "socket_timeout": 30,
        "retries": 5,
        "fragment_retries": 5,
        # Politeness / stability
        "quiet": True,
        "no_warnings": True,
    }
    opts.update(get_cookie_opt())
    return opts

# ---------- Helper: locate ytagent's output file ----------
def find_ytagent_output(search_dir="."):
    """
    ytagent writes downloads/<video_id>.mp4 by default.
    We tell it to use a dedicated folder and then find the newest .mp4 in it.
    """
    candidate_dirs = [
        os.path.join(search_dir, "ytagent_downloads"),
        os.path.join(search_dir, "downloads"),
        search_dir,
    ]
    newest = None
    newest_mtime = 0
    for d in candidate_dirs:
        if not os.path.isdir(d):
            continue
        for fname in os.listdir(d):
            if fname.lower().endswith(".mp4"):
                fpath = os.path.join(d, fname)
                mtime = os.path.getmtime(fpath)
                if mtime > newest_mtime:
                    newest = fpath
                    newest_mtime = mtime
    return newest

# ---------- Search ----------
def on_search_clicked(b):
    query = search_input.value.strip()
    if not query:
        return

    with output_area:
        clear_output()
        print("📡 Searching...")

        ydl_opts = {
            "extract_flat": True,
            **get_base_opts(),
        }

        dropdown_options = []

        try:
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                info = ydl.extract_info(
                    f"ytsearch20:{query}",
                    download=False
                )

            for idx, entry in enumerate(info.get("entries", [])):
                if not entry:
                    continue

                title = entry.get("title", "Unknown Video")
                video_url = entry.get("webpage_url")

                if not video_url and entry.get("id"):
                    video_url = f"https://www.youtube.com/watch?v={entry['id']}"

                if video_url:
                    label = f"{idx + 1}. {title[:70]}"
                    dropdown_options.append((label, video_url))

            if dropdown_options:
                track_select.options = dropdown_options
                track_select.disabled = False
                run_button.disabled = False
                print("✅ Select a video and click Download & Play.")
            else:
                print("❌ No videos found.")

        except Exception as e:
            print(f"❌ Search error: {e}")

# ---------- Download (with ytagent fallback) ----------
def on_execute_clicked(b):
    video_url = track_select.value

    with output_area:
        clear_output()
        print("📥 Downloading with yt-dlp...")

        target_file = "downloaded_video.mp4"

        if os.path.exists(target_file):
            os.remove(target_file)

        ydl_opts = {
            "format": (
                "bestvideo[height<=480][ext=mp4]+"
                "bestaudio[ext=m4a]/"
                "best[height<=480]/best"
            ),
            "outtmpl": "downloaded_video.%(ext)s",
            "merge_output_format": "mp4",
            "noplaylist": True,
            "quiet": False,
            **get_base_opts(),
        }
        ydl_opts["quiet"] = False  # show progress / errors

        ytdlp_failed = False
        try:
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                ydl.download([video_url])
        except Exception as e:
            ytdlp_failed = True
            print(f"⚠️ yt-dlp failed: {e}")

        # ----- Fallback: ytagent -----
        if ytdlp_failed or not os.path.exists(target_file):
            print("\n🔁 Falling back to ytagent...")

            ytagent_dir = "ytagent_downloads"
            if os.path.exists(ytagent_dir):
                shutil.rmtree(ytagent_dir, ignore_errors=True)
            os.makedirs(ytagent_dir, exist_ok=True)

            try:
                result = subprocess.run(
                    [
                        "ytagent", "download", video_url,
                        "--out-dir", ytagent_dir,
                    ],
                    capture_output=True, text=True, timeout=600
                )
                if result.stdout:
                    print(result.stdout[-2000:])  # show last part of ytagent log
                if result.returncode != 0:
                    print(f"❌ ytagent failed: {result.stderr[-1000:]}")
                else:
                    found = find_ytagent_output(".")
                    if found and os.path.getsize(found) > 5000:
                        shutil.move(found, target_file)
                        print(f"✅ ytagent downloaded: {target_file}")
                    else:
                        print("❌ ytagent ran but no valid .mp4 was found.")
            except FileNotFoundError:
                print("❌ ytagent not installed. Add `ytagent-cli` to requirements.txt and rebuild.")
            except subprocess.TimeoutExpired:
                print("❌ ytagent timed out after 10 minutes.")
            except Exception as e:
                print(f"❌ ytagent error: {e}")

        # ----- Display result -----
        if os.path.exists(target_file) and os.path.getsize(target_file) > 5000:
            print("\n🎉 Download complete!")
            display(Video(target_file, embed=True, width=640))
            display(FileLink(target_file))
        else:
            print("\n❌ No playable video file was produced.")

# ---------- Wire up events ----------
search_button.on_click(on_search_clicked)

def on_enter(change):
    if change["name"] == "value" and change["new"].endswith("\n"):
        search_input.value = change["new"].rstrip("\n")
        on_search_clicked(None)

search_input.continuous_update = False
search_input.observe(on_enter, names="value")

run_button.on_click(on_execute_clicked)

# ---------- Display UI ----------
display(widgets.VBox([
    widgets.HBox([search_input, search_button]),
    widgets.HBox([track_select, run_button]),
    output_area
]))